In [1]:
%pip install requests pandas openpyxl argparse
import getpass
import argparse
import json
import os
import re
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import requests

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# STEP 1: Settings

In [2]:
HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "gemma4-26b"
TEMPERATURE = 0           # 0 = deterministic / predictable
SEED = 123
MAX_WORKERS = 2              # Parallel calls; lower it if the server returns errors
TIMEOUT = 180                # Seconds to wait for one answer
MAX_RETRIES = 3              # Tries per paragraph before it is marked as an error
STOP_AFTER_ERRORS = 20       # Pause the whole run if this many fail in a row

# Prompt for API Key
API_KEY = os.environ.get("UNI_API_KEY") or getpass.getpass("Paste the Uni API key: ")
print("Key loaded:", bool(API_KEY))

Key loaded: True


# STEP 2: The Annotation Prompt

In [3]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")
# cold explicit
# Customize your labels and definition here
ANNOTATION_PROMPT = """
ROLE You are a text annotator for media content analysis. Apply the Stereotype Content Model (SCM) to decide whether [Gruppe 1] is portrayed with Cold/Coldness (the opposite of Warmth) in German newspaper articles. CONTEXT Input: several consecutive paragraphs from one article plus the target group [Gruppe 1]. Return one label for the whole passage. Entity names are masked as [Gruppe 1] and [andere Gruppe]; keep them exactly as written, also in your explanation. TASK Label TRUE if the text portrays [Gruppe 1] or a member of [Gruppe 1] as the ACTOR of a cold trait or cold behavior toward others, shown by an explicit marker. Otherwise label FALSE. Others (the receiver) can be anyone: [andere Gruppe], individuals, the public, or [Gruppe 1]'s own group. Membership must be stated or unmistakable in the text. If you have to guess that someone belongs to [Gruppe 1], they do not count. Pronouns (they, their) count when the antecedent in the passage is clearly [Gruppe 1]: in "They are calling on their citizens to attack our cities", "They" counts if the previous sentence names [Gruppe 1]. Others may comment on or accuse [Gruppe 1] (see QUOTES), but the cold trait or behavior must be attributed to [Gruppe 1] or a member as the actor. Judge as a general reader: do not decide whether the author or a quoted speaker is right, only what the text conveys about [Gruppe 1]. DEFINITION Coldness = perceived negative traits or negative behavior of [Gruppe 1] toward others, in two dimensions: Unfriendliness: traits such as unfriendly, cold, unsociable, aloof, unpleasant, hostile or rude (but not limited to these), or treating others in such a way. Immorality: traits such as immoral, untrustworthy, dishonest, insincere, unfair, intolerant, selfish or malicious (but not limited to these), or treating others dishonestly, unfairly, exploitatively or harmfully. EXPLICIT MARKERS (a) A cold trait word (or synonym / noun form) applied to [Gruppe 1] or a member, e.g. "rude", "their dishonesty". (b) A concretely described cold act whose hostile, dishonest, unfair or selfish nature is stated or unmistakable (see COLD BEHAVIOR). (c) An explicit negative moral evaluation, e.g. "selfishly hoarded resources". (d) Emotive or hyperbolic framing of crime, violence or terrorism (see CRIME). EVIDENCE LEVEL: EXPLICIT ONLY Only explicit markers (a) to (d) count. Implicit cues (tone, insinuation, loaded wording without a trait or act, hyperbole or emotive wording on its own outside crime reporting, calls to action whose hostile aim is only implied, stereotypes, guilt by association, hints at bad motives) are NOT coded: if they are the only evidence, S1 is F. COLD BEHAVIOR A behavior counts only if it is a real action with a tangible negative effect or clearly hostile intent: cheating or exploiting others; lying to or betraying them; harming, attacking, threatening or harassing them; calling on others to attack or harm them (when the hostile aim is stated) or inciting against a group; excluding or discriminating without legitimate reason; sabotaging their interests, property or reputation; hoarding resources they clearly need. It does not count if it is trivial (not greeting, being reserved, declining an invitation, a curt remark), disagreement (criticizing, protesting, defending one's own legitimate interests), cultural, religious or lifestyle difference, or an act with no evident hostile, dishonest, unfair or selfish intent. CRIME, VIOLENCE AND TERRORISM (overrides COLD BEHAVIOR) A purely factual report (who, what, when, where, charges, police or court information) is not coldness: "The [Gruppe 1] national was charged with arson on Monday afternoon." Emotive labels, hyperbole, moral condemnation, attributed malicious intent or fear appeals aimed at [Gruppe 1] or a member are coldness, also inside quotes: "These [Gruppe 1] terrorists are stabbing people at night, so avoid going into this area at night." or "[Gruppe 1] gangs are turning the city into a war zone." Hyperbole (exaggerating how malicious or dangerous [Gruppe 1] or a member is) and emotive wording are what turn a report into coldness. If a factual and an emotive sentence appear for the same acts, the emotive sentence decides. Emotion about the victims or the situation alone does not count. Outside crime reporting, hyperbole or emotive wording about [Gruppe 1] is explicit only when it is attached to a stated cold act, trait or moral evaluation (markers (a) to (c)); on its own it is an implicit cue, so NOT coded. QUOTES Coldness may come from the source or from any quoted speaker (including [andere Gruppe] or [Gruppe 1] itself), if it attributes the cold trait or behavior to [Gruppe 1] or a member as the actor. It does not count when the cold word describes someone else: "Filip is rude to everyone," says the [Gruppe 1]. NOT COLDNESS Step 2 (actor): others are cold toward [Gruppe 1]; [Gruppe 1] is only a victim ("Three [Gruppe 1] were attacked"); the cold trait or behavior belongs to someone not identifiable as [Gruppe 1]; [Gruppe 1] is a place or geographic location. Step 3 (content): competence or incompetence ("cowardly", "lazy"); threat frame only ("[Gruppe 1] weakens the economy"); trivial behavior or disagreement; purely factual crime report. DECISION RULES Answer all four steps for the passage, strictly in this order, and decide each one from the text before moving to the next. The explanation is built from these steps, never the other way around: do not form a summary or a label first and then fit the steps to it. Write the Summary only after S1, S2, S3 and IMPLICIT are decided and the CONFIDENCE triggers are checked.
MARKER: Is there an explicit marker (a) to (d)? No if there is none, including facts and statistics only or implicit cues only.
ACTOR: Is [Gruppe 1] or a member, the one/group who is perceived to have cold character traits or described as acting cold toward others? No if a Step 2 item of NOT COLDNESS applies.
NOT EXCLUDED: Is it true that no Step 3 item of NOT COLDNESS applies?
CONFIDENCE: The label is TRUE if S1, S2 and S3 are all T, otherwise FALSE. For a TRUE label, check the triggers below. CONFIDENCE A TRUE label needs an exact span from the passage that shows [Gruppe 1] or a member as the cold actor. If you cannot point to one, label FALSE. If a TRUE label has at least one trigger, keep TRUE and start the explanation with "NOT SURE:" and the trigger codes. With no trigger, do not hedge. T1 Dual reading: the key word may mean coldness or only toughness or competence ("ruthless", "tough", "strict"). T2 Crime wording: borderline whether a crime or attack report is factual or emotive/hyperbolic. T3 Borderline explicit/implicit: it is borderline whether the evidence is an explicit marker or only implicit. Never use NOT SURE on FALSE. Unclear membership is FALSE, not NOT SURE. OUTPUT FORMAT Return only a valid JSON object, with no text before or after it. The object has exactly two keys, in this order: "explanation": one string in this form: S1: T/F | S2: T/F | S3: T/F | IMPLICIT: T/F | Summary: ...
Write the fields strictly in this order. First decide S1, S2, S3 and IMPLICIT by working through the DECISION RULES, then check the CONFIDENCE triggers, and only then write the Summary. The Summary justifies the fields and must not contradict them.
S1: a marker was found. S2: [Gruppe 1] or a member is the actor. S3: no Step 3 exclusion applies. IMPLICIT: T if the strongest evidence is implicit, F if it is explicit or there is none.
Summary: one short sentence naming the actor and the key words from the text in single quotes. If a CONFIDENCE trigger applies, start the Summary with "NOT SURE: T1" (all trigger codes that apply) and then the sentence.
Fill all four fields even when the label is FALSE. Use single quotes only inside the string. "label": "TRUE" if S1, S2 and S3 are all T, otherwise "FALSE".
EXAMPLES Passage: The [Gruppe 1] owner deliberately ruined a rival shop by spreading lies about it. {"explanation": "S1: T | S2: T | S3: T | IMPLICIT: F | Summary: The [Gruppe 1] owner 'deliberately ruined a rival shop by spreading lies'.", "label": "TRUE"} Passage: Colleagues described the [Gruppe 1] manager as ruthless in negotiations. {"explanation": "S1: T | S2: T | S3: T | IMPLICIT: F | Summary: NOT SURE: T1 Colleagues call the [Gruppe 1] manager 'ruthless', which may mean tough rather than cold.", "label": "TRUE"} Passage: The local population cheated [Gruppe 1] out of their share. {"explanation": "S1: T | S2: F | S3: T | IMPLICIT: F | Summary: The local population 'cheated [Gruppe 1]', so the coldness is toward [Gruppe 1], not from it.", "label": "FALSE"} Passage: The [Gruppe 1] national was charged with arson on Monday afternoon. {"explanation": "S1: T | S2: T | S3: F | IMPLICIT: F | Summary: A [Gruppe 1] national 'charged with arson', reported purely factually.", "label": "FALSE"}
"""

instructions = ANNOTATION_PROMPT
reminder = "Ensure you strictly evaluate only the explicit coldness attributes of [Gruppe 1] within the text. Do not code coldness based on implicit cues or on the actions of other groups"
json_mode = True

print("--- Instructions to annotate COLD (EXPLICIT) Loaded ---")
json_mode= True


--- Instructions to annotate COLD (EXPLICIT) Loaded ---


# STEP 3: Paragraph Input Data

In [4]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

INPUT_EXCEL_FILE = "sample_5k_translated.xlsx"
CHECKPOINT_FILE = "cold_e_annotation_checkpoint.jsonl"
OUTPUT_EXCEL = "sample_5k_cold_e_annotated.xlsx"
OUTPUT_JSON = "sample_5k_cold_e_annotated.json"

try:
    sample = pd.read_excel(INPUT_EXCEL_FILE)
    if "row_id" not in sample.columns:
        sample.insert(0, "row_id", range(len(sample)))

    # Preprocessing: Exclude the group 'JUDE' from input data
    initial_len = len(sample)
    group_col = None
    for col in ["group", "target_group", "target", "group_name", "Gruppe", "gruppe"]:
        if col in sample.columns:
            group_col = col
            break

    if group_col:
        sample = sample[sample[group_col].astype(str).str.strip().str.upper() != "JUDE"].reset_index(drop=True)
        print(f"Preprocessing: Excluded {initial_len - len(sample)} rows where '{group_col}' was 'JUDE'.")
    else:
        for col in sample.select_dtypes(include=["object"]).columns:
            sample = sample[sample[col].astype(str).str.strip().str.upper() != "JUDE"]
        sample = sample.reset_index(drop=True)
        print(f"Preprocessing: Excluded 'JUDE' across text columns ({initial_len - len(sample)} rows removed).")

    print(f"Successfully loaded {len(sample)} paragraph(s) from '{INPUT_EXCEL_FILE}'.")
except FileNotFoundError:
    raise FileNotFoundError(f"Error: Could not find '{INPUT_EXCEL_FILE}' in the current folder.")

Preprocessing: Excluded 189 rows where 'group' was 'JUDE'.
Successfully loaded 4811 paragraph(s) from 'sample_5k_translated.xlsx'.


# STEP 4: Set up LLM API Call Function

In [5]:
if "API_KEY" not in globals():
    raise RuntimeError("Please run STEP 1 first.")

def call_llm(text, instructions, reminder=None, max_retries=MAX_RETRIES):
    user_msg = f"Text: '{text}'"
    if reminder:
        user_msg += f"\nReminder: {reminder}"
        
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": instructions},
            {"role": "user",   "content": user_msg},
        ],
        "temperature": TEMPERATURE,
        "seed": SEED,
        "reasoning_effort": "none",
        "response_format": {"type": "json_object"},
        "max_tokens": 1000,
    }
    
    last_error = None
    if json_mode:
        body["response_format"] = {"type": "json_object"}

        
    for attempt in range(1, max_retries + 1):
        try:
            r = requests.post(f"{HOST}/chat/completions",
                              headers={"Authorization": f"Bearer {API_KEY}"},
                              json=body, timeout=TIMEOUT)
                                             
            r.raise_for_status()
            j = r.json()
            return {
                "output": j["choices"][0]["message"]["content"].strip(),
                "tokens_prompt": j["usage"]["prompt_tokens"],
                "tokens_response": j["usage"]["completion_tokens"],
                "error": None,
            }
            
        except requests.exceptions.HTTPError as he:
            last_error = str(he)
            if he.response.status_code == 504:
                # 504 Gateway Timeout: back off aggressively 
                wait_time = 15 * (2 ** (attempt - 1))
                print(f"Attempt {attempt}/{max_retries} failed: 504 Gateway Timeout. Server is overloaded. Waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"Attempt {attempt}/{max_retries} failed with HTTP error: {he}")
                time.sleep(5 * attempt)
                # Back off softly if too many requests are sent

            if r.status_code == 429:
                                wait_time = 60
                                print(f"Attempt {attempt}/{max_retries} failed: 429 Too Many Requests. Cooling down for {wait_time}s...")
                                time.sleep(wait_time)
                                continue

            if he.response.status_code is not None and 400 <= he.response.status_code < 500 and he.response.status_code not in (408, 429):
                break

        except Exception as e:
            last_error = str(e)
            print(f"Attempt {attempt}/{max_retries} failed (Network error): {e}")
            time.sleep(5 * attempt)
            
    print("All retry attempts failed for this entry.")
    return {"output": None, "tokens_prompt": None, "tokens_response": None, "error": last_error}

# STEP 5: Run Annotation Loop with parallelization 

In [ ]:
if "API_KEY" not in globals() or "sample" not in globals():
    raise RuntimeError("Please run STEP 1 and STEP 3 first.")

print(f"Starting annotation loop for {len(sample)} entries...")

# 1. Load existing checkpoints if any (enables resuming if interrupted)
completed_ids = set()
results_cache = {}

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                row_id = data.get("row_id")
                if row_id is not None:
                    completed_ids.add(row_id)
                    results_cache[row_id] = data
            except json.JSONDecodeError:
                continue
    print(f"Resumed from checkpoint: {len(completed_ids)} already annotated entries found.")

# Thread-safe lock for writing to the checkpoint file
file_lock = threading.Lock()

def process_row(row):
    row_id = row["row_id"]
    if row_id in completed_ids:
        return results_cache[row_id]
    
    # Explicitly grab text from the 'translation' column
    text_content = str(row.get("translation", ""))
    
    response = call_llm(text_content, instructions, reminder=reminder)
    
    # Parse the LLM output JSON string
    parsed_output = {"label": None, "explanation": "Failed to parse JSON"}
    
    if response["output"]:
        try:
            clean_out = response["output"].strip()
            if clean_out.startswith("```json"):
                clean_out = clean_out[7:]
            if clean_out.startswith("```"):
                clean_out = clean_out[3:]
            if clean_out.endswith("```"):
                clean_out = clean_out[:-3]
            
            parsed_json = json.loads(clean_out.strip())
            parsed_output["label"] = parsed_json.get("label")
            parsed_output["explanation"] = parsed_json.get("explanation")
        except Exception as e:
            parsed_output["explanation"] = f"JSON parse error: {e} | Raw: {response['output']}"

    result_entry = {
        "row_id": row_id,
        "label": parsed_output["label"],
        "explanation": parsed_output["explanation"],
        "raw_output": response["output"],
        "error": response["error"],
        "tokens_prompt": response["tokens_prompt"],
        "tokens_response": response["tokens_response"]
    }
    
    with file_lock:
        with open(CHECKPOINT_FILE, "a", encoding="utf-8") as cp:
            cp.write(json.dumps(result_entry, ensure_ascii=False) + "\n")
            
    return result_entry

# 2. Run execution using ThreadPoolExecutor
results = []
errors_in_a_row = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_row, row): row for _, row in sample.iterrows()}
    
    completed_count = len(completed_ids)
    total_count = len(sample)
    
    for future in as_completed(futures):
        res = future.result()
        results.append(res)
        completed_count += 1
        
        if res.get("error"):
            errors_in_a_row += 1
        else:
            errors_in_a_row = 0
            
        if errors_in_a_row >= STOP_AFTER_ERRORS:
            print(f"Stopping early: {STOP_AFTER_ERRORS} errors encountered in a row.")
            break
            
        if completed_count % 10 == 0 or completed_count == total_count:
            print(f"Progress: {completed_count}/{total_count} processed.")

# 3. Merge results back into DataFrame and save final outputs
results_df = pd.DataFrame(results)
final_df = pd.merge(sample, results_df, on="row_id", how="left")

final_df.to_excel(OUTPUT_EXCEL, index=False)
final_df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)

print("\n--- Annotation Complete! ---")
print(f"Checkpoint saved to: {CHECKPOINT_FILE}")
print(f"Excel output saved to: {OUTPUT_EXCEL}")
print(f"JSON output saved to: {OUTPUT_JSON}")

Starting annotation loop for 4811 entries...
Progress: 10/4811 processed.
Progress: 20/4811 processed.
Progress: 30/4811 processed.
Progress: 40/4811 processed.
Progress: 50/4811 processed.
Progress: 60/4811 processed.
Progress: 70/4811 processed.
Progress: 80/4811 processed.
Progress: 90/4811 processed.
Progress: 100/4811 processed.
Progress: 110/4811 processed.
Progress: 120/4811 processed.
Progress: 130/4811 processed.
Progress: 140/4811 processed.
Progress: 150/4811 processed.
Progress: 160/4811 processed.
Progress: 170/4811 processed.
Progress: 180/4811 processed.
Progress: 190/4811 processed.
Progress: 200/4811 processed.
Progress: 210/4811 processed.
Progress: 220/4811 processed.
Progress: 230/4811 processed.
Progress: 240/4811 processed.
Progress: 250/4811 processed.
Progress: 260/4811 processed.
Progress: 270/4811 processed.
Progress: 280/4811 processed.
Progress: 290/4811 processed.
Progress: 300/4811 processed.
Progress: 310/4811 processed.
Progress: 320/4811 processed.
Prog